In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
device = torch.device('cuda:0')
import random

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='train_log.log', level=logging.INFO)

from Dataset import ImportanceDataset, RealImportanceDataset, RealPredictionDataset, XBoxDatasetSimulation
from GAN import GAN
from Discriminator import DataDiscriminator
from util import set_seed
import itertools
import pandas as pd
#optimize over w directly, dont use any theta shenannigans. 
from sklearn import preprocessing
import pytorch_warmup as warmup
from DataProcessing import *


In [ ]:
#creating bias xbox and gt
GT_SIZE = 10000
BIAS_SIZE = 1000

test = XBoxDatasetSimulation("./data/gcHouse,7attributes.csv")
def save_new_XBOX_csvs():
    global_GT_dict, global_GT_var_order,global_bias_dict,global_bias_var_order = XBOX_get_GT_and_bias_ratios()

    GT_persons_count = get_all_persons_types_count(GT_SIZE,global_GT_dict)
    BT_persons_count = get_all_persons_types_count(BIAS_SIZE,global_bias_dict)

    GT_sampled_df = XBOX_get_sampled_df(global_GT_var_order,
                                        GT_persons_count,
                                        test.df)
    bias_sampled_df = XBOX_get_sampled_df(global_bias_var_order,
                                        BT_persons_count,
                                        test.df)

    bias_df_normalized, bias_NaN_columns = normalize_df(bias_sampled_df)
    gt_df_normalized, gt_df_NaN_columns = normalize_df(GT_sampled_df)
    all_NaNs_columns = bias_NaN_columns and gt_df_NaN_columns

    clean_NaN_by_col_index(bias_df_normalized,all_NaNs_columns)
    clean_NaN_by_col_index(gt_df_normalized,all_NaNs_columns)

    print(bias_df_normalized.shape, gt_df_normalized.shape)
    #save to CSV files
    bias_df_normalized = bias_df_normalized.sample(frac=1).reset_index(drop=True)
    gt_df_normalized = gt_df_normalized.sample(frac=1).reset_index(drop=True)

    bias_df_normalized.to_csv(BIAS_SAVE_PATH + "XBOX_bias.csv",index=False)
    gt_df_normalized.to_csv(GT_SAVE_PATH + "XBOX_GT.csv",index=False)


In [ ]:
#create new dataset 

target_var_diff = 0.2

from DataProcessing import *

df = pd.read_csv("./data/usa_00004.csv")
df = clean_NaN_target_col(df)
columns_with_NaN = list(df.isna().any())
clean_NaN(df, columns_with_NaN)
print("Done")

gt_df, bias_df, gt_df_target_mean, bias_df_target_mean = force_sample_from_df(df,GT_size=GT_SIZE, 
                                                                              BIAS_size=BIAS_SIZE,
                                                                              target_var_diff=target_var_diff)

#normalize bias and gt data set 
bias_df_normalized, bias_NaN_columns = normalize_df(bias_df)
gt_df_normalized, gt_df_NaN_columns = normalize_df(gt_df)
all_NaNs_columns = bias_NaN_columns and gt_df_NaN_columns

#purge any columns with NaN that are introduced as a result of normalization
clean_NaN(bias_df_normalized, all_NaNs_columns)
clean_NaN(gt_df_normalized, all_NaNs_columns)

print(bias_df_normalized.shape, gt_df_normalized.shape)
#save to CSV files
bias_df_normalized.to_csv(BIAS_SAVE_PATH + "TargetVar="+str(bias_df_target_mean)+
                            ",NumPoints:"+str(BIAS_SIZE)+".csv",index=False)
gt_df_normalized.to_csv(GT_SAVE_PATH + "TargetVar="+str(gt_df_target_mean)+
                            ",NumPoints:"+str(GT_SIZE)+".csv",index=False)




In [ ]:
#NEW TRAINING VARIABLE SET UP
# For learning

BIAS_BATCH_SIZE = 100
TRUTH_BATCH_SIZE = BIAS_BATCH_SIZE
GENERATOR_TRAINING_FACTOR = 1
DISCRIMINATOR_TRAINING_FACTOR = 1

TEMPERATURE_START = 0.1
TEMPERATURE_END = 0.1
TEMPERATURE = 0.1
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# For dataset
num_totaldata_ground_truth = 1000
zero_prob = 0

seeds = []
for _ in range(10):
    seeds.append(np.random.randint(1e6))
print('bias batch size: ', BIAS_BATCH_SIZE)
print('truth batch size: ', TRUTH_BATCH_SIZE)
print("Seeds: ", seeds)

In [ ]:
%autoreload 2
GENERATOR_LEARNING_RATE = 5e-5
DISCRIMINATOR_LEARNING_RATE = 1e-4
BATCHS_IN_EPOCH = 1
EPOCHS = 15000  # the stream is infinite so one epoch will be defined as BATCHS_IN_EPOCH * BATCH_SIZE

layers = [256] #[256 for _ in range(5)]

generator_types = ['dataGen'] #['dataGen','weightsGen','onesGen']

warmup_durations = [4000]
for seed in seeds:
    set_seed(seed)
    if False:
        d = ImportanceDataset(ground_truth_path=None,
                        num_biased_data_points=num_biased_data_points,
                        device=device)
        d = RealImportanceDataset(ground_truth_path='./data/normalizedCleaned100.csv',
                                num_totaldata_ground_truth=1000,
                                device=device)
    save_new_XBOX_csvs()
    d = RealPredictionDataset(ground_truth_path='./data/XBOX_GT.csv',
                              bias_path = './data/XBOX_bias.csv',
                              device=device)
    for var in warmup_durations:
        gan = GAN(
            dataset=d,
            generator_type = 'dataGen',
            gen_learning_rate=GENERATOR_LEARNING_RATE,
            disc_learning_rate=DISCRIMINATOR_LEARNING_RATE,
            truth_sample_size=TRUTH_BATCH_SIZE,
            gen_layers=layers,
            bias_sample_size=BIAS_BATCH_SIZE,
            temperature=0.1,
            warmup_length = var,
            )
        writer = SummaryWriter(comment='||gen_lr='+str(GENERATOR_LEARNING_RATE)+
                               'disc_lr='+str(DISCRIMINATOR_LEARNING_RATE)+
                               '||var:'+str(var)+'||seed:'+str(seed))
        
        clusters, probs, prob_diffs,  test_probs, test_prob_diffs, generator_losses, discriminator_losses  = gan.train(BATCHS_IN_EPOCH,
                                                                                                                    EPOCHS,
                                                                                                                    TEMPERATURE_START,
                                                                                                                    TEMPERATURE_END,
                                                                                                                    GENERATOR_TRAINING_FACTOR,
                                                                                                                    DISCRIMINATOR_TRAINING_FACTOR,
                                                                                                                    writer)

    biased_weights = gan.generator.get_weights(d.biased_dataset).flatten()
    biased_avg_attributes = d.biased_dataset.cpu().numpy().mean(axis=0)
    biased_learned_attributes = biased_weights @ d.biased_dataset.cpu().numpy()
    GT_avg_attributes = d.ground_truth_dataset.cpu().numpy().mean(axis=0)

    logger.info('biased_avg_attributes: ' + str(biased_avg_attributes))
    logger.info('GT_avg_attributes: ' + str(GT_avg_attributes))
    logger.info('biased_learned_attributes: ' + str(biased_learned_attributes))
    logger.info('difference per: ' + str(abs(biased_learned_attributes - GT_avg_attributes)))
    logger.info('difference avg: ' + str(np.mean(abs(biased_learned_attributes - GT_avg_attributes))))
    logger.info('---------------------------------------')

In [ ]:
class DummyGenerator(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()

        self.weights = nn.Parameter(torch.rand((1, 10), requires_grad=True))

    def forward(self, tensor_dataset):
        indexes = F.gumbel_softmax(self.weights.repeat(2,1),tau=0.1,hard=False)
        return indexes @ tensor_dataset
d = ImportanceDataset(ground_truth_path=None,
                    sample_size=BIAS_BATCH_SIZE,
                    batch_size=1,)

gan = GAN(bias_dataset=d.biased_data,
            ground_truth_dataset=d.ground_truth_dataset,
            ground_truth_prob=d.ground_truth_prob,
            learning_rate=LEARNING_RATE,
            truth_batch_size=TRUTH_BATCH_SIZE,
            num_features=num_features,
            gen_layers=None,
            bias_batch_size=BIAS_BATCH_SIZE,
            temperature=0.1,
            )

In [ ]:
model = DummyGenerator()
model.to(torch.device('cuda:0'))

In [ ]:
output = model(gan.bias_dataset)

In [ ]:
#code that tests how well a discriminator operates

def assess_discriminator(discriminator, 
                         discriminator_optim,
                         dataset, 
                         groundtruth_weights, 
                         biased_weights,
                         Loss_function,
                         subset_size=2,
                         num_epochs = 1000,
                         device=torch.device('cuda:0'),
                         ):
    '''
    Given the input, it assesses how well a discriminator object can differentiate between two weightings of a singular data set. 

    Returns two loss np arrays in the form of a dictionary, results:
        results['truth'] = np array
        results['bias] = np array

    Input:
        discriminator - discriminator object
        discriminator_optim - discriminator optimizer
        dataset - np array size (n x d), where n = number of points, d = dimensionality
        groundtruth_weights - np array, sum(weights_one) = 1, groundtruth_weights.shape[0] = n 
        biased_weights - np array, sum(weights_two) = 1, biased_weights.shape[0] = n 
        subset_size - int, the size of the sample to take from the data set
        num_epochs - number of training iterations
    
    Output: results - dict
    '''
    results = {}
    results['truth'] = np.zeros(num_epochs)
    results['bias'] = np.zeros(num_epochs)
    for ne in range(num_epochs):
        #get data sample using groundtruth_weights
        groundtruth_indexes = np.random.choice(np.arange(dataset.shape[0]),size=subset_size,p=groundtruth_weights)
        groundtruth_onehot = np.zeros((subset_size, dataset.shape[0]))
        groundtruth_onehot[np.arange(subset_size),groundtruth_indexes] = 1
        groundtruth_sample = torch.tensor((groundtruth_onehot @ dataset),dtype=torch.float32).to(device).flatten()

        #get data sample using biased_weights
        biased_indexes = np.random.choice(np.arange(dataset.shape[0]),size=subset_size,p=biased_weights)
        biased_onehot = np.zeros((subset_size, dataset.shape[0]))
        biased_onehot[np.arange(subset_size),biased_indexes] = 1
        biased_sample = torch.tensor((biased_onehot @ dataset),dtype=torch.float32).to(device).flatten()

        #pass both from discriminator and generate losses
        truth_prediction = discriminator(groundtruth_sample)
        biased_prediction = discriminator(biased_sample)
        loss_real = Loss_function(truth_prediction.to(device), torch.ones_like(truth_prediction).to(device))
        loss_fake = Loss_function(biased_prediction.to(device), torch.zeros_like(biased_prediction).to(device))
        discriminator_loss = loss_real+loss_fake
        #update discriminator
        discriminator_optim.zero_grad()
        discriminator_loss.backward(retain_graph=True)
        discriminator_optim.step()
        #store loss in arrays
        results['truth'][ne] = loss_real.detach().cpu().item()
        results['bias'][ne] = loss_real.detach().cpu().item()
    return results

SUBSET_SIZE = 10
DATA_DIM = 10
discriminator = DataDiscriminator(subset_size=SUBSET_SIZE,data_dimension=DATA_DIM)
discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
L = nn.BCEWithLogitsLoss()

loss_dict = assess_discriminator(discriminator, 
                     discriminator_optimizer,
                    dataset.groundtruth_data, 
                    groundtruth_weights=[0.1 for _ in range(dataset.n_truth)], 
                    biased_weights=[0.1 for _ in range(dataset.n_truth)],
                    Loss_function = L,
                    subset_size=SUBSET_SIZE,
                    num_epochs = 100,
                    device=device,
                    )